In [3]:
import pandas as pd
import seaborn as sns
import numpy as np

# Load the Titanic dataset
titanic_df = sns.load_dataset('titanic')

# Display the first few rows and some info
print("Titanic Dataset Head:")
display(titanic_df.head())
print("\nTitanic Dataset Info:")
titanic_df.info()

Titanic Dataset Head:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True



Titanic Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Prepare the data
# Drop columns that are highly correlated or have many missing values for simplicity
# Also drop 'deck' as it has many NaNs and 'embark_town' as 'embarked' covers it
titanic_cleaned = titanic_df.drop(columns=['who', 'adult_male', 'class', 'embark_town', 'deck', 'alive', 'alone'])

# Handle missing values: fill 'age' with median, 'embarked' with mode
titanic_cleaned['age'] = titanic_cleaned['age'].fillna(titanic_cleaned['age'].median())
titanic_cleaned['embarked'] = titanic_cleaned['embarked'].fillna(titanic_cleaned['embarked'].mode()[0])
titanic_cleaned['fare'] = titanic_cleaned['fare'].fillna(titanic_cleaned['fare'].median())

# Convert categorical features to numerical using Label Encoding
le = LabelEncoder()
for column in ['sex', 'embarked']: # 'survived' is already 0/1
    titanic_cleaned[column] = le.fit_transform(titanic_cleaned[column])

# Define target and potential features
X = titanic_cleaned.drop('survived', axis=1)
y = titanic_cleaned['survived']

# Let's drop name and passengerid as they are identifiers
X = X.drop(columns=['pclass', 'sibsp', 'parch', 'ticket', 'fare', 'age'], errors='ignore') # We'll build up features from here

potential_features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Features available for greedy selection:", potential_features)

best_accuracy = 0
current_features = []

while True:
    improvement_found = False
    candidate_features = list(set(potential_features) - set(current_features))

    if not candidate_features: # No more features to add
        break

    best_candidate_for_this_step = None
    best_accuracy_for_this_step = best_accuracy

    for feature in candidate_features:
        test_features = current_features + [feature]

        # Use only the selected features for training
        X_train_selected = titanic_cleaned[test_features].iloc[X_train.index]
        X_test_selected = titanic_cleaned[test_features].iloc[X_test.index]

        model = RandomForestClassifier(random_state=42)
        model.fit(X_train_selected, y_train)
        predictions = model.predict(X_test_selected)
        accuracy = accuracy_score(y_test, predictions)

        if accuracy > best_accuracy_for_this_step:
            best_accuracy_for_this_step = accuracy
            best_candidate_for_this_step = feature
            improvement_found = True

    if improvement_found:
        current_features.append(best_candidate_for_this_step)
        best_accuracy = best_accuracy_for_this_step
        print(f"Added feature: '{best_candidate_for_this_step}'. Current features: {current_features}. Best accuracy: {best_accuracy:.4f}")
    else:
        break # No improvement, stop adding features

print("\n--- Greedy Feature Selection Complete ---")
print("Final selected features:", current_features)
print(f"Highest accuracy achieved: {best_accuracy:.4f}")

Features available for greedy selection: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
Added feature: 'sex'. Current features: ['sex']. Best accuracy: 0.7910
Added feature: 'sibsp'. Current features: ['sex', 'sibsp']. Best accuracy: 0.7985

--- Greedy Feature Selection Complete ---
Final selected features: ['sex', 'sibsp']
Highest accuracy achieved: 0.7985
